In [34]:
import polars as pl
import pandas as pd
import numpy as np
import datetime as dt
import os
import sys
import json 
import importlib

sys.path.append("../utils/")

import helpers as hp

In [35]:
rng = np.random.default_rng(seed=274)

In [36]:

with open("../configs/zone_distances.json", "r") as json_file:
    zone_distances = json.load(json_file)

In [37]:
zone_distances

{'Yerbabuena': {'centroids': [20.964404421308334, -101.2847459818324],
  'points': [{'point': [20.9743, -101.300626],
    'distance': 0.01871089133756122},
   {'point': [20.946278, -101.298483], 'distance': 0.022743632462390598},
   {'point': [20.951706, -101.264939], 'distance': 0.023527992541503697},
   {'point': [20.979447, -101.274624], 'distance': 0.018131014585796808},
   {'point': [20.985938, -101.285636], 'distance': 0.02155196379935847},
   {'point': [20.9743, -101.300626], 'distance': 0.01871089133756122}]},
 'Marfil': {'centroids': [20.998877268148007, -101.29015919031652],
  'points': [{'point': [20.979212, -101.292779],
    'distance': 0.019839006379117525},
   {'point': [20.991242, -101.304647], 'distance': 0.016376628136369357},
   {'point': [21.006043, -101.294156], 'distance': 0.008205010702043177},
   {'point': [21.019272, -101.283363], 'distance': 0.021497285645706604},
   {'point': [21.008196, -101.275124], 'distance': 0.017688858391175503},
   {'point': [20.999863,

In [38]:
gym_hours = {"open" : 6,
             "close" : 22}

profiles = {
"1": {
"name" : "frequent_morning",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .69},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .56},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .75},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .57},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" : .71},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .76},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : 6.43,
"hour_std" : .75,
"plan_probability" : {"1": 0.05, "2": 0.2, "3" : 0.45, "4": 0.3}
},
"2" : {
"name" : "frequent_noon",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .67},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .62},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .57},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .67},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.43},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .74},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : 13.16,
"hour_std" : 1.40,
"plan_probability" : {"1": .2, "2": 0.5, "3" : 0.25, "4": 0.05}
            },
"3" :{
"name" : "frequent_night",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .54},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .77},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .6},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .84},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.4},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .58},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : 19.5,
"hour_std" : 1.15,
"plan_probability" : {"1": 0.15, "2": 0.15, "3" : 0.45, "4": 0.25}
},
"4" :{
"name" : "random_morning",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .4},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .4},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .3},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : 0.5},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" : .6},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .65},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : 8.5,
"hour_std" : 2.05,
"plan_probability" : {"1": 0.65, "2": 0.25, "3" : 0.1, "4": 0}
},
"5" :{
"name" : "random_night",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .3},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .3},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .5},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .6},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" : .59},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .3},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : 20.1,
"hour_std" : 2.08,
"plan_probability" : {"1": 0.4, "2": 0.3, "3" : 0.2, "4": 0.1}
},
"6" :{
"name" : "full_random",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .5},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .5},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .5},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .5},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.5},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .5},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : 14.00,
"hour_std" : 3.0,
"plan_probability" : {"1": 0.59, "2": 0.3, "3" : 0.1, "4": 0.01}
},
"7" :{
"name" : "rare_random",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .2},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .2},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .4},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .2},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.3},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .4},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : 16.0,
"hour_std" : 3.5,
"plan_probability" : {"1": 0.85, "2": 0.1, "3" : 0.05, "4": 0}
},
"8" :{
"name" : "ultrarare_random",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .1},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .2},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .1},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .1},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.2},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .3},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : 14.5,
"hour_std" : 5.5,
"plan_probability" : {"1": 0.9, "2": 0.08, "3" : 0.02, "4": 0}
}
}

profile_weights = {"1" : 0.23076923, "2": 0.12820513, "3" : 0.25641026, "4" : 0.07692308, "5" : 0.07692308, "6" : 0.05128205, "7" : 0.07692308, "8" : 0.1025641}

for profile in profiles.keys():
    print(profile)
    days = profiles[profile]["days_probability"]

    sum_weight = sum([days[day]["day_weight"] for day in days.keys()])
        
    for day in days.keys():
        days[day]["day_probability"] = days[day]["day_weight"] /sum_weight
    
    profiles[profile]["total_weight"] = sum_weight

    assert 1 == sum([i for i in profiles[profile]["plan_probability"].values() ])
profiles

1
2
3
4
5
6
7
8


{'1': {'name': 'frequent_morning',
  'days_probability': {'mon': {'day_number': 1,
    'day_name': 'Monday',
    'day_weight': 0.69,
    'day_probability': 0.17079207920792078},
   'tue': {'day_number': 2,
    'day_name': 'Tuesday',
    'day_weight': 0.56,
    'day_probability': 0.13861386138613863},
   'wed': {'day_number': 3,
    'day_name': 'Wednesday',
    'day_weight': 0.75,
    'day_probability': 0.18564356435643564},
   'thu': {'day_number': 4,
    'day_name': 'Thursday',
    'day_weight': 0.57,
    'day_probability': 0.14108910891089108},
   'fri': {'day_number': 5,
    'day_name': 'Friday',
    'day_weight': 0.71,
    'day_probability': 0.17574257425742573},
   'sat': {'day_number': 6,
    'day_name': 'Saturday',
    'day_weight': 0.76,
    'day_probability': 0.18811881188118812},
   'sun': {'day_number': 7,
    'day_name': 'Sunday',
    'day_weight': 0,
    'day_probability': 0.0}},
  'hour_mean': 6.43,
  'hour_std': 0.75,
  'plan_probability': {'1': 0.05, '2': 0.2, '3': 0.45

In [39]:
# Functions
def create_customer_profiles(profile_weights_ : dict, n_):
    return {str(i): str(int(rng.choice( list(profile_weights.keys()), p = list(profile_weights.values()) ) ) ) for i in range(n_) }

In [40]:
create_customer_profiles(profiles, 9)

{'0': '8',
 '1': '1',
 '2': '3',
 '3': '7',
 '4': '3',
 '5': '4',
 '6': '1',
 '7': '5',
 '8': '1'}

In [41]:
# prob = np.array((.9,0.5,1,.3,.3,.2,.3,.4))

# prob/sum(prob)

In [42]:
importlib.reload(hp)

oli =hp.create_customer(zone_distances)

Creating customer
36254


In [43]:
oli

{'id': None,
 'name': 'Alicia Sofía',
 'last_name': 'Alva Sisneros',
 'gender': np.False_,
 'birth_date': datetime.date(1993, 9, 7),
 'lat': 20.98821795202615,
 'lon': -101.29917077337164,
 'zipcode': 36254,
 'email': 'g**************r@gmail.com',
 'phone_number': '(473)8821424',
 'created_at': datetime.datetime(2026, 6, 3, 17, 7, 3, 166508),
 'status': 'Active',
 'profile_type': None,
 'updated_at': datetime.datetime(2026, 6, 3, 17, 7, 3, 166513)}

In [44]:
# customers_schema = {"Customer_Id" :pl.Int64,
#                     "Name": pl.String,
#                     "Last_Name": pl.String,
#                     "Gender" : pl.Boolean,
#                     "Birth_Date" : pl.Datetime,
#                     "Latitude": pl.Float32,
#                     "Longitude" : pl.Float32,
#                     "Zipcode" : pl.Int64 ,
#                     "Email" : pl.String,
#                     "Phone_Number" : pl.String,
#                     "Created_At" : pl.Date,
#                     "Status" : pl.String,
#                     "Profile_Type" : pl.String,
#                     "Updated_At" : pl.Datetime}

# df_customers = pl.read_csv("../data/customers_v2.csv", schema=customers_schema)


In [45]:
# df_customers.with_columns(pl.col("Birth_Date").cast(pl.Date)).write_csv("../data/customers_v2.csv")

In [ ]:
def create_customer_access_data(date_, profile_metadata_, gym_hours_, data_path_ = "../data/", customers_file_ = "customers.csv", payment_file_ = "payments.csv", access_file_ = "access.csv", plans_file_ = "suscription_plans.csv"):
    
    opening_date = "2026-01-02"

    assert date_ >= opening_date, f"Gym doesn't exists before {opening_date}"

    
    # Define Schemas
    customers_schema = {"Customer_Id" :pl.Int64,
                        "Name": pl.String,
                        "Last_Name": pl.String,
                        "Gender" : pl.Boolean,
                        "Birth_Date" : pl.Date,
                        "Latitude": pl.Float32,
                        "Longitude" : pl.Float32,
                        "Zipcode" : pl.Int64 ,
                        "Email" : pl.String,
                        "Phone_Number" : pl.String,
                        "Created_At" : pl.Date,
                        "Status" : pl.String,
                        "Profile_Type" : pl.String,
                        "Updated_At" : pl.Datetime}
    

    payments_schema = {"Payment_Id" : pl.String,
                       "Plan_Id" : pl.Int64,
                       "Customer_Id" : pl.Int64,
                       "Payment_Amount" : pl.Float16,
                       "Payment_Time" : pl.Datetime,
                       "Plan_Expiration_Time" : pl.Datetime,
                       "Payment_Status": pl.String,
                       "Updated_At" : pl.Datetime
    }

    access_schema = {"Visit_Id" : pl.Int64,
                     "Customer_Id" : pl.Int64,
                     "Plan_Id" : pl.Int64,
                     "Branch_Id" : pl.Int8,
                     "Payment_Id" : pl.String,
                     "Date" : pl.Date,
                     "Visit_Time" : pl.Datetime,
                     "Updated_At" : pl.Datetime
    }

    suscription_plan_schema = {"Plan_Id" : pl.Int64,
                                "Plan_Name" : pl.String,
                                "Plan_Cost" : pl.Float16, 
                                "Plan_Duration" : pl.Int64,
                                "Plan_Start_Time" : pl.Datetime, 
                                "Plan_End_Time" : pl.Datetime, 
                                "Plan_Description" :pl.String,
                                "Updated_At": pl.Datetime
                                }
    
    eval_date = dt.datetime.strptime(date_, "%Y-%m-%d")
    day_names_dic = {0 : 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4:'Friday', 5:'Saturday', 6: 'Sunday'}
    day_of_week = eval_date.weekday()
    day_name = day_names_dic[day_of_week]

    print(eval_date, day_of_week, day_name)
    
    if not os.path.isdir(data_path_):
        os.makedirs(data_path_, exist_ok = True)

    if not os.path.exists(data_path_ + customers_file_):
        print("Creating customers file")
        df_customers = pl.DataFrame([], schema = customers_schema,
                                                    orient="row").write_csv(data_path_+ customers_file_)
    
    if not os.path.exists(data_path_ + payment_file_):
        print("Creating payments file")
        df_payments = pl.DataFrame([], schema = payments_schema,
                                                    orient="row").write_csv(data_path_+ payment_file_)
        
    if not os.path.exists(data_path_ + access_file_):
        print("Creating access registry file")
        df_access = pl.DataFrame([], schema = access_schema,
                                                    orient="row").write_csv(data_path_+ access_file_)

    df_customers = pl.read_csv(data_path_+ customers_file_, schema=customers_schema)
    df_access = pl.read_csv(data_path_+ access_file_, schema=access_schema)
    df_plans = pl.read_csv(data_path_ + plans_file_, schema = suscription_plan_schema)
    df_payments = pl.read_csv(data_path_+ payment_file_, schema=payments_schema)
    
    #########################
    # 1. Create new customers
    #########################

    # 1.1. Customers configs
    tot_new = int(abs(rng.normal(0, 1)))

    max_id = df_customers.select(pl.max("Customer_Id")).item()

    print(max_id)
    counter = max_id
    if max_id == None:
        counter = 0
        tot_new = 12

    print("Total new customers", tot_new)
    counter += 1 

    dict_profiles = create_customer_profiles(profile_metadata_, tot_new)

    # 1.2 Generate customers
    new_customers = []
    new_customers_ids = []
    for new_cus in range(tot_new):

        new_customer = hp.create_customer(zone_distances)
        new_customer["id"] = counter
        new_customer["profile_type"] = dict_profiles[str(new_cus)]
        new_customer["created_at"] = dt.datetime.strptime(date_, "%Y-%m-%d").date()

        new_customers.append(tuple(new_customer.values()))
        new_customers_ids.append(counter)
        counter +=1

    # 1.3 Save file
    df_new_customers = pl.DataFrame(new_customers, customers_schema, orient = "row")
    df_customers = pl.concat([df_customers, df_new_customers])
    df_customers.write_csv(data_path_ + customers_file_)

    ###########################
    # 2. Create Access Registry
    ###########################

    # 2.1 Given at least one customer, generate random visits
    if df_customers.shape[0] >0:
        day_cus = df_customers.filter(pl.col("Status") =="Active").to_dicts()

        
        payments = []
        access = []
        
        access_count = df_access.select(pl.max("Visit_Id")).item()
        if access_count == None:
            access_count = 0

        for row in day_cus:
            cus_id = row["Customer_Id"]
            profile = str(row["Profile_Type"])
            profile_data = profile_metadata_[profile]
            day_proba  = profile_data["days_probability"][day_name[0:3].lower()]["day_weight"]
            print(day_proba)

            has_visit = rng.choice([1,0], p = [day_proba, 1-day_proba])

            if max_id == None:
                has_visit = 1

            customer_new = "old customer"
            if cus_id in new_customers_ids: # If it's a new customer, has to enter gym
                has_visit = 1
                customer_new = "new_customer"

            print(f"({customer_new}) Customer {cus_id} with profile {profile} appeared {bool(has_visit)}")

            # Check is customer assisted
            if has_visit == 0:
                continue
            
            ######################
            # 2.2. Create Payments
            ######################

            #Check if plan is still active
            temp_pay = df_payments.filter(pl.col("Customer_Id") == cus_id).sort(by = ["Plan_Expiration_Time"], descending=[True])
            #display(temp_pay)
            payment_count = temp_pay.shape[0]

            
            expiration_date = temp_pay.select(pl.max("Plan_Expiration_Time")).item()

            while True:
                arrival_time = rng.normal(profile_data["hour_mean"], profile_data["hour_std"])
                if (arrival_time >= gym_hours_['open']) and (arrival_time < gym_hours_["close"]-1):
                    break
            
            arrival_hour = int(arrival_time)
            arrival_minute = int(arrival_time % arrival_hour * 60)
            
            arrival_time_obj = dt.time(arrival_hour, arrival_minute, rng.integers(0,59))

            plan_choice = 0
            #display(temp_pay.head(1))
            
            last_pay = temp_pay.head(1)
            if last_pay.is_empty() == False:
                plan_choice = last_pay.select(pl.col("Plan_Id")).item()
                print("checando plan_choice", plan_choice)

            momentum_pass_visits = 0
            if plan_choice == 2:
                payment_id = df_access.filter(pl.col("Customer_Id") == cus_id).sort(by = ["Visit_Id"], descending=[True]).head(1).select(pl.col("Payment_Id")).item()
                momentum_pass_visits += df_access.filter(pl.col("Payment_Id") == payment_id).shape[0]
            # Generate payment if conditions are meet

            print(payment_count, expiration_date, momentum_pass_visits)
            if (payment_count == 0) or (expiration_date < dt.datetime.combine(dt.datetime.strptime(date_, "%Y-%m-%d"), arrival_time_obj)) or (momentum_pass_visits >= 5) :
                
                plan_choice = int(rng.choice( list(profile_metadata_[profile]["plan_probability"].keys()), p= list(profile_metadata_[profile]["plan_probability"].values())))

                if cus_id in new_customers_ids: # If it's a new customer, has to enter gym
                    plan_choice = 1

                print(f"Customer {cus_id} is paying plan {plan_choice}")

                temp_plan  = df_plans.filter(pl.col("Plan_Id")== plan_choice)

                amount = temp_plan.select(pl.col("Plan_Cost")).item()
                duration = temp_plan.select(pl.col("Plan_Duration")).item()
                payment_count +=1
                payments_dict = {"Payment_Id" : str(cus_id) + "-" + str(payment_count),
                                "Plan_Id" : plan_choice,
                                "Customer_Id" : cus_id,
                                "Payment_Amount" : amount,
                                "Payment_Time" : dt.datetime.combine(dt.datetime.strptime(date_, "%Y-%m-%d"), arrival_time_obj),
                                "Plan_Expiration_Time" : dt.datetime.strptime(date_, "%Y-%m-%d").replace(hour = 23, minute= 59, second = 59) + dt.timedelta(days = duration -1),
                                "Payment_Status": "Completed",
                                "Updated_At" : dt.datetime.combine(dt.datetime.strptime(date_, "%Y-%m-%d"), arrival_time_obj)
                }

                payments.append(tuple(payments_dict.values()))

            
                # df_new_payments = pl.DataFrame(payments, schema = payments_schema, orient = "row")
                # df_payments = pl.concat([df_payments, df_new_payments])
                # df_payments.write_csv(data_path_+ payment_file_)
            
            # Generate access
            df_access = pl.read_csv(data_path_+ access_file_, schema=access_schema)

            visit_time = rng.normal(profile_data["hour_mean"], profile_data["hour_std"])

            # if plan_choice  == None:
            #      plan_choice = temp_pay.select(pl.col("Plan_Id")).head(1).item()
            #      print("checando plan_choice", plan_choice)

            access_dict = {"Visit_Id" : access_count + 1,
                    "Customer_Id" : cus_id,
                    "Plan_Id" : plan_choice,
                    "Branch_Id" : 1,
                    "Payment_Id" : str(cus_id) + "-" + str(payment_count),
                    "Date" : dt.datetime.strptime(date_, "%Y-%m-%d").date(),
                    "Visit_Time" : dt.datetime.combine(dt.datetime.strptime(date_, "%Y-%m-%d"), arrival_time_obj),
                    "Updated_At" : dt.datetime.combine(dt.datetime.strptime(date_, "%Y-%m-%d"), arrival_time_obj)
                    }

            access_count += 1

            access.append(tuple(access_dict.values()))

    df_new_accesss = pl.DataFrame(access, schema = access_schema, orient = "row")
    df_access = pl.concat([df_access, df_new_accesss])
    df_access.write_csv(data_path_+ access_file_)

    df_new_payments = pl.DataFrame(payments, schema = payments_schema, orient = "row")
    df_payments = pl.concat([df_payments, df_new_payments])
    df_payments.write_csv(data_path_+ payment_file_)

    return df_customers, df_payments, df_access
    

In [47]:
[day.strftime("%Y-%m-%d") for day in pd.date_range(start= "2026-01-02", end = "2026-01-10")]

['2026-01-02',
 '2026-01-03',
 '2026-01-04',
 '2026-01-05',
 '2026-01-06',
 '2026-01-07',
 '2026-01-08',
 '2026-01-09',
 '2026-01-10']

In [ ]:
for m_day in [day.strftime("%Y-%m-%d") for day in pd.date_range(start= "2026-05-31", end = "2026-06-03")]:

    iter_date = dt.datetime.strptime(day, "%Y-%m-%d")
    
    if iter_date.weekday() == 6:
        print("Momentum doesn't open on Sundays")
        continue

    datasets = create_customer_access_data(m_day, profiles, gym_hours, customers_file_ = "customers_v2.csv", payment_file_ = "payments_v2.csv", access_file_ = "access_v2.csv" )


display(datasets[0])
display(datasets[1])
display(datasets[2])

2026-05-31 00:00:00 6 Sunday
49
Total new customers 2
Creating customer
36013
Creating customer
36253
0
(old customer) Customer 1 with profile 3 appeared False
0
(old customer) Customer 2 with profile 1 appeared False
0
(old customer) Customer 3 with profile 1 appeared False
0
(old customer) Customer 4 with profile 5 appeared False
0
(old customer) Customer 5 with profile 1 appeared False
0
(old customer) Customer 6 with profile 8 appeared False
0
(old customer) Customer 7 with profile 4 appeared False
0
(old customer) Customer 8 with profile 5 appeared False
0
(old customer) Customer 9 with profile 1 appeared False
0
(old customer) Customer 10 with profile 7 appeared False
0
(old customer) Customer 11 with profile 4 appeared False
0
(old customer) Customer 12 with profile 3 appeared False
0
(old customer) Customer 13 with profile 1 appeared False
0
(old customer) Customer 14 with profile 1 appeared False
0
(old customer) Customer 15 with profile 3 appeared False
0
(old customer) Custo

Customer_Id,Name,Last_Name,Gender,Birth_Date,Latitude,Longitude,Zipcode,Email,Phone_Number,Created_At,Status,Profile_Type,Updated_At
i64,str,str,bool,date,f32,f32,i64,str,str,date,str,str,datetime[μs]
1,"""Ernesto Silvano""","""Arroyo Mora""",true,2000-12-18,21.024862,-101.25441,36013,"""z************m@hotmail.com""","""(473)7977952""",2026-02-01,"""Active""","""3""",2026-06-01 21:40:09.390294
2,"""Óliver Mauricio""","""Saiz de la Garza""",true,2004-08-20,21.017761,-101.254082,36000,"""r********g@gmail.com""","""(473)7554933""",2026-02-01,"""Active""","""1""",2026-06-01 21:40:12.797359
3,"""Eva Amalia""","""Flores Rangel""",false,2000-07-09,21.002136,-101.257401,36080,"""v*********h@gmail.com""","""(473)3150401""",2026-02-01,"""Active""","""1""",2026-06-01 21:40:16.135513
4,"""Joaquín Alejandro""","""Mercado Espinoza""",true,2006-05-28,21.03986,-101.247597,36023,"""e**********o@gmail.com""","""(473)1597298""",2026-02-01,"""Active""","""5""",2026-06-01 21:40:19.694481
5,"""Dulce Tania""","""Villaseñor Echeverría""",false,1972-09-03,21.018347,-101.249214,36003,"""t********l@gmail.com""","""(473)1420050""",2026-02-01,"""Active""","""1""",2026-06-01 21:40:23.149214
…,…,…,…,…,…,…,…,…,…,…,…,…,…
47,"""Teresa Eugenia""","""Ordóñez Gálvez""",false,2002-01-16,21.022772,-101.237778,36093,"""v*********u@gmail.com""","""(473)9350054""",2026-05-27,"""Active""","""8""",2026-06-01 21:42:43.289621
48,"""Felix René""","""Reyes Covarrubias""",true,2005-09-13,20.96137,-101.279083,36259,"""k*********y@hotmail.com""","""(473)3671901""",2026-05-27,"""Active""","""2""",2026-06-01 21:42:46.862664
49,"""Sofía Nadia""","""Alba Herrera""",false,2006-06-15,21.03676,-101.256516,36023,"""u*************i@gmail.com""","""(473)7941183""",2026-05-29,"""Active""","""2""",2026-06-01 21:42:50.318468


Payment_Id,Plan_Id,Customer_Id,Payment_Amount,Payment_Time,Plan_Expiration_Time,Payment_Status,Updated_At
str,i64,i64,f16,datetime[μs],datetime[μs],str,datetime[μs]
"""1-1""",1,1,150.0,2026-02-01 19:45:11,2026-02-01 23:59:59,"""Completed""",2026-02-01 19:45:11
"""2-1""",1,2,150.0,2026-02-01 06:19:00,2026-02-01 23:59:59,"""Completed""",2026-02-01 06:19:00
"""3-1""",1,3,150.0,2026-02-01 06:53:56,2026-02-01 23:59:59,"""Completed""",2026-02-01 06:53:56
"""4-1""",1,4,150.0,2026-02-01 17:11:39,2026-02-01 23:59:59,"""Completed""",2026-02-01 17:11:39
"""5-1""",1,5,150.0,2026-02-01 07:08:30,2026-02-01 23:59:59,"""Completed""",2026-02-01 07:08:30
…,…,…,…,…,…,…,…
"""50-2""",3,50,1200.0,2026-06-02 19:07:50,2026-07-01 23:59:59,"""Completed""",2026-06-02 19:07:50
"""51-2""",4,51,10000.0,2026-06-02 06:40:32,2027-06-01 23:59:59,"""Completed""",2026-06-02 06:40:32
"""10-21""",1,10,150.0,2026-06-03 14:18:08,2026-06-03 23:59:59,"""Completed""",2026-06-03 14:18:08


Visit_Id,Customer_Id,Plan_Id,Branch_Id,Payment_Id,Date,Visit_Time,Updated_At
i64,i64,i64,i8,str,date,datetime[μs],datetime[μs]
1,1,1,1,"""1-1""",2026-02-01,2026-02-01 19:45:11,2026-02-01 19:45:11
2,2,1,1,"""2-1""",2026-02-01,2026-02-01 06:19:00,2026-02-01 06:19:00
3,3,1,1,"""3-1""",2026-02-01,2026-02-01 06:53:56,2026-02-01 06:53:56
4,4,1,1,"""4-1""",2026-02-01,2026-02-01 17:11:39,2026-02-01 17:11:39
5,5,1,1,"""5-1""",2026-02-01,2026-02-01 07:08:30,2026-02-01 07:08:30
…,…,…,…,…,…,…,…
1710,42,1,1,"""42-10""",2026-06-03,2026-06-03 08:56:56,2026-06-03 08:56:56
1711,45,2,1,"""45-2""",2026-06-03,2026-06-03 13:55:57,2026-06-03 13:55:57
1712,48,3,1,"""48-2""",2026-06-03,2026-06-03 11:41:13,2026-06-03 11:41:13


In [49]:
display(datasets[0])
display(datasets[1])
display(datasets[2])

Customer_Id,Name,Last_Name,Gender,Birth_Date,Latitude,Longitude,Zipcode,Email,Phone_Number,Created_At,Status,Profile_Type,Updated_At
i64,str,str,bool,date,f32,f32,i64,str,str,date,str,str,datetime[μs]
1,"""Ernesto Silvano""","""Arroyo Mora""",true,2000-12-18,21.024862,-101.25441,36013,"""z************m@hotmail.com""","""(473)7977952""",2026-02-01,"""Active""","""3""",2026-06-01 21:40:09.390294
2,"""Óliver Mauricio""","""Saiz de la Garza""",true,2004-08-20,21.017761,-101.254082,36000,"""r********g@gmail.com""","""(473)7554933""",2026-02-01,"""Active""","""1""",2026-06-01 21:40:12.797359
3,"""Eva Amalia""","""Flores Rangel""",false,2000-07-09,21.002136,-101.257401,36080,"""v*********h@gmail.com""","""(473)3150401""",2026-02-01,"""Active""","""1""",2026-06-01 21:40:16.135513
4,"""Joaquín Alejandro""","""Mercado Espinoza""",true,2006-05-28,21.03986,-101.247597,36023,"""e**********o@gmail.com""","""(473)1597298""",2026-02-01,"""Active""","""5""",2026-06-01 21:40:19.694481
5,"""Dulce Tania""","""Villaseñor Echeverría""",false,1972-09-03,21.018347,-101.249214,36003,"""t********l@gmail.com""","""(473)1420050""",2026-02-01,"""Active""","""1""",2026-06-01 21:40:23.149214
…,…,…,…,…,…,…,…,…,…,…,…,…,…
47,"""Teresa Eugenia""","""Ordóñez Gálvez""",false,2002-01-16,21.022772,-101.237778,36093,"""v*********u@gmail.com""","""(473)9350054""",2026-05-27,"""Active""","""8""",2026-06-01 21:42:43.289621
48,"""Felix René""","""Reyes Covarrubias""",true,2005-09-13,20.96137,-101.279083,36259,"""k*********y@hotmail.com""","""(473)3671901""",2026-05-27,"""Active""","""2""",2026-06-01 21:42:46.862664
49,"""Sofía Nadia""","""Alba Herrera""",false,2006-06-15,21.03676,-101.256516,36023,"""u*************i@gmail.com""","""(473)7941183""",2026-05-29,"""Active""","""2""",2026-06-01 21:42:50.318468


Payment_Id,Plan_Id,Customer_Id,Payment_Amount,Payment_Time,Plan_Expiration_Time,Payment_Status,Updated_At
str,i64,i64,f16,datetime[μs],datetime[μs],str,datetime[μs]
"""1-1""",1,1,150.0,2026-02-01 19:45:11,2026-02-01 23:59:59,"""Completed""",2026-02-01 19:45:11
"""2-1""",1,2,150.0,2026-02-01 06:19:00,2026-02-01 23:59:59,"""Completed""",2026-02-01 06:19:00
"""3-1""",1,3,150.0,2026-02-01 06:53:56,2026-02-01 23:59:59,"""Completed""",2026-02-01 06:53:56
"""4-1""",1,4,150.0,2026-02-01 17:11:39,2026-02-01 23:59:59,"""Completed""",2026-02-01 17:11:39
"""5-1""",1,5,150.0,2026-02-01 07:08:30,2026-02-01 23:59:59,"""Completed""",2026-02-01 07:08:30
…,…,…,…,…,…,…,…
"""50-2""",3,50,1200.0,2026-06-02 19:07:50,2026-07-01 23:59:59,"""Completed""",2026-06-02 19:07:50
"""51-2""",4,51,10000.0,2026-06-02 06:40:32,2027-06-01 23:59:59,"""Completed""",2026-06-02 06:40:32
"""10-21""",1,10,150.0,2026-06-03 14:18:08,2026-06-03 23:59:59,"""Completed""",2026-06-03 14:18:08


Visit_Id,Customer_Id,Plan_Id,Branch_Id,Payment_Id,Date,Visit_Time,Updated_At
i64,i64,i64,i8,str,date,datetime[μs],datetime[μs]
1,1,1,1,"""1-1""",2026-02-01,2026-02-01 19:45:11,2026-02-01 19:45:11
2,2,1,1,"""2-1""",2026-02-01,2026-02-01 06:19:00,2026-02-01 06:19:00
3,3,1,1,"""3-1""",2026-02-01,2026-02-01 06:53:56,2026-02-01 06:53:56
4,4,1,1,"""4-1""",2026-02-01,2026-02-01 17:11:39,2026-02-01 17:11:39
5,5,1,1,"""5-1""",2026-02-01,2026-02-01 07:08:30,2026-02-01 07:08:30
…,…,…,…,…,…,…,…
1710,42,1,1,"""42-10""",2026-06-03,2026-06-03 08:56:56,2026-06-03 08:56:56
1711,45,2,1,"""45-2""",2026-06-03,2026-06-03 13:55:57,2026-06-03 13:55:57
1712,48,3,1,"""48-2""",2026-06-03,2026-06-03 11:41:13,2026-06-03 11:41:13


In [50]:
datasets[1].filter(pl.col("Customer_Id") == 4)

Payment_Id,Plan_Id,Customer_Id,Payment_Amount,Payment_Time,Plan_Expiration_Time,Payment_Status,Updated_At
str,i64,i64,f16,datetime[μs],datetime[μs],str,datetime[μs]
"""4-1""",1,4,150.0,2026-02-01 17:11:39,2026-02-01 23:59:59,"""Completed""",2026-02-01 17:11:39
"""4-2""",3,4,1200.0,2026-02-04 18:17:44,2026-03-05 23:59:59,"""Completed""",2026-02-04 18:17:44
"""4-3""",3,4,1200.0,2026-03-06 16:40:42,2026-04-04 23:59:59,"""Completed""",2026-03-06 16:40:42
"""4-4""",1,4,150.0,2026-04-09 19:42:42,2026-04-09 23:59:59,"""Completed""",2026-04-09 19:42:42
"""4-5""",1,4,150.0,2026-04-10 18:36:06,2026-04-10 23:59:59,"""Completed""",2026-04-10 18:36:06
"""4-6""",2,4,650.0,2026-04-13 19:31:47,2026-05-12 23:59:59,"""Completed""",2026-04-13 19:31:47
"""4-7""",2,4,650.0,2026-04-22 20:51:22,2026-05-21 23:59:59,"""Completed""",2026-04-22 20:51:22
"""4-8""",4,4,10000.0,2026-05-01 20:18:23,2027-04-30 23:59:59,"""Completed""",2026-05-01 20:18:23


In [51]:
datasets[2].filter(pl.col("Customer_Id") == 4)

Visit_Id,Customer_Id,Plan_Id,Branch_Id,Payment_Id,Date,Visit_Time,Updated_At
i64,i64,i64,i8,str,date,datetime[μs],datetime[μs]
4,4,1,1,"""4-1""",2026-02-01,2026-02-01 17:11:39,2026-02-01 17:11:39
24,4,3,1,"""4-2""",2026-02-04,2026-02-04 18:17:44,2026-02-04 18:17:44
36,4,3,1,"""4-2""",2026-02-06,2026-02-06 19:53:40,2026-02-06 19:53:40
43,4,3,1,"""4-2""",2026-02-07,2026-02-07 18:23:53,2026-02-07 18:23:53
50,4,3,1,"""4-2""",2026-02-09,2026-02-09 19:28:47,2026-02-09 19:28:47
…,…,…,…,…,…,…,…
1582,4,4,1,"""4-8""",2026-05-29,2026-05-29 19:06:51,2026-05-29 19:06:51
1606,4,4,1,"""4-8""",2026-05-30,2026-05-30 18:45:47,2026-05-30 18:45:47
1638,4,4,1,"""4-8""",2026-06-01,2026-06-01 20:59:43,2026-06-01 20:59:43


In [52]:
datasets[2].group_by("Customer_Id", "Plan_Id").len().sort(by = "len", descending=True)

Customer_Id,Plan_Id,len
i64,i64,u32
2,4,71
5,4,68
12,3,67
1,4,66
20,4,62
…,…,…
28,1,1
33,1,1
50,1,1
